<a href="https://colab.research.google.com/github/UnkindledTwo/perceptual_backbone_ldct/blob/main/perceptual_backbone_ldct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download and unzip dataset
!gdown 1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y&export=download
!unzip dataset_biailab_I2I_2025v1.zip

Downloading...
From (original): https://drive.google.com/uc?id=1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y
From (redirected): https://drive.google.com/uc?id=1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y&confirm=t&uuid=fc2760ac-8720-47b9-84bb-7e6aa7958c42
To: /content/dataset_biailab_I2I_2025v1.zip
100% 5.74G/5.74G [01:16<00:00, 74.9MB/s]
Archive:  dataset_biailab_I2I_2025v1.zip
   creating: minideeplesion/
  inflating: __MACOSX/._minideeplesion  
   creating: minideeplesion/test/
  inflating: __MACOSX/minideeplesion/._test  
   creating: minideeplesion/train/
  inflating: __MACOSX/minideeplesion/._train  
   creating: minideeplesion/val/
  inflating: __MACOSX/minideeplesion/._val  
  inflating: minideeplesion/test/001286_01_02_049.png.npz  
  inflating: __MACOSX/minideeplesion/test/._001286_01_02_049.png.npz  
  inflating: minideeplesion/test/001276_01_02_161.png.npz  
  inflating: __MACOSX/minideeplesion/test/._001276_01_02_161.png.npz  
  inflating: minideeplesion/test/001276_04_02_173.png.npz  
  inflati

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Clone repo and add to path
!git clone https://github.com/UnkindledTwo/perceptual_backbone_ldct

Cloning into 'perceptual_backbone_ldct'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 22 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 42.31 KiB | 2.01 MiB/s, done.


In [4]:
import torch
from pathlib import Path

import sys
sys.path.append("/content/perceptual_backbone_ldct/src")
sys.path.append("/content/perceptual_backbone_ldct")

from config import PerceptualConfig
CONFIG = PerceptualConfig(run_name="screening-c0", data_root=Path("/content/minideeplesion/"))

In [5]:
from pathlib import Path
from data import assert_no_patient_leakage
patient_leakage = assert_no_patient_leakage(Path(CONFIG.data_root))
print("Patients per split:",patient_leakage)

Patients per split: {'train': 115, 'val': 34, 'test': 31}


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

train_file_path = CONFIG.data_root.as_posix() + '/train/*.npz'
test_file_path = CONFIG.data_root.as_posix() + '/test/*.npz'
val_file_path = CONFIG.data_root.as_posix() + '/val/*.npz'

# Find all files
train_files = sorted(glob(train_file_path))
test_files = sorted(glob(test_file_path))
val_files = sorted(glob(val_file_path))

print(f"Training samples: {len(train_files)}")
print(f"Test samples: {len(test_files)}")
print(f"Eval samples: {len(val_files)}")

# Load a single file
data = np.load(train_files[0])
print(f"Available arrays: {data.files}")

# Access the arrays
gt_image = data[CONFIG.target_key]
noisy_image = data[CONFIG.input_key]

print(f"Image shape: {gt_image.shape}")
print(f"Value range: [{gt_image.min():.3f}, {gt_image.max():.3f}]")

Training samples: 899
Test samples: 220
Eval samples: 182
Available arrays: ['gt_full', 'gt_clipped', 'noisy_clipped_10k', 'noisy_clipped_50k', 'noisy_clipped_100k', 'noisy_clipped_500k', 'noisy_clipped_1000k']
Image shape: (512, 512)
Value range: [0.000, 1.000]


In [7]:
from data import I2IPairs
from torch.utils.data import DataLoader

# Dataloaders
splits = {s: I2IPairs(Path(CONFIG.data_root / s), CONFIG.input_key, CONFIG.target_key) for s in ("train", "val", "test")}
train_loader = DataLoader(splits["train"], batch_size=CONFIG.batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(splits["test"], batch_size=CONFIG.batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(splits["val"], batch_size=CONFIG.batch_size, shuffle=True, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 113
Test batches: 28
Val batches: 23


In [8]:
!pip install lpips piq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 2.1 MB/s eta 0:00:00


In [9]:
# Identity test for psnr, ssim, lpips-alex, dists and R_h
from metrics import slice_metrics, save_per_slice_metrics

noisy, gt, _ = next(iter((test_loader)))

noisy_2d = noisy[0].squeeze().cpu().numpy()
gt_2d = gt[0].squeeze().cpu().numpy()

# Run the metrics on this single slice
metrics_noisy = slice_metrics(noisy_2d, gt_2d)
metrics_gt = slice_metrics(gt_2d, gt_2d)

print(metrics_noisy)
print(metrics_gt)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:00<00:00, 266MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
Downloading: "https://github.com/photosynthesis-team/piq/releases/download/v0.4.1/dists_weights.pt" to /root/.cache/torch/hub/checkpoints/dists_weights.pt


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/skimage/metrics/simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


{'psnr': 21.963722129341964, 'ssim': 0.729700554974769, 'lpips': 0.35036760568618774, 'dists': 0.18659216165542603, 'r_h': 8.296412367742322}
{'psnr': inf, 'ssim': 1.0, 'lpips': 0.0, 'dists': 1.1920928955078125e-07, 'r_h': 1.0}


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from model import UNet

# Init U-Net

model = UNet().to(device)
model.eval()

UNet(
  (encoder): ModuleList(
    (0): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
      )
    )
    (1): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace

In [ ]:
from train import train_and_test
import time
import random

print("Seed:", CONFIG.seed)
random.seed(CONFIG.seed)

criterion = torch.nn.functional.l1_loss

start_time = time.time()
df_test = train_and_test(config=CONFIG, model=model, criterion=criterion, device=device)
end_time = time.time()
total_time = end_time - start_time
print("Total time", total_time)
print(f"Total training time: {total_time:.2f} seconds")

Seed: 42


screening-c0: 100%|██████████| 30/30 [06:10<00:00, 12.33s/it]


Total time 390.9660849571228
Total training time: 390.97 seconds


In [ ]:
summary = {
    "psnr_pred_mean":  float(df_test["psnr"].mean()),
    "psnr_pred_std":   float(df_test["psnr"].std(ddof=1)),
    "ssim_pred_mean":  float(df_test["ssim"].mean()),
    "ssim_pred_std":   float(df_test["ssim"].std(ddof=1)),
    "lpips_pred_mean":  float(df_test["lpips"].mean()),
    "lpips_pred_std":   float(df_test["lpips"].std(ddof=1)),
    "dists_pred_mean":  float(df_test["dists"].mean()),
    "dists_pred_std":   float(df_test["dists"].std(ddof=1)),
    "r_h_pred_mean":  float(df_test["r_h"].mean()),
    "r_h_pred_std":   float(df_test["r_h"].std(ddof=1)),
    "n_test": int(len(df_test)),
    "total_time": total_time,
}

print("\n=== TEST RESULTS (mean \u00b1 std) ===")
print(f"Model  PSNR: {summary['psnr_pred_mean']:.2f} \u00b1 {summary['psnr_pred_std']:.2f} dB")
print(f"Model  SSIM: {summary['ssim_pred_mean']:.4f} \u00b1 {summary['ssim_pred_std']:.4f}")
print(f"Model  LP\u0130PS: {summary['lpips_pred_mean']:.4f} \u00b1 {summary['lpips_pred_std']:.4f}")
print(f"Model  DISTS: {summary['dists_pred_mean']:.4f} \u00b1 {summary['dists_pred_std']:.4f}")
print(f"Model  R_H: {summary['r_h_pred_mean']:.4f} \u00b1 {summary['r_h_pred_std']:.4f}")

print(f"Test samples: {summary['n_test']}")

save_per_slice_metrics([summary], CONFIG.run_dir / "summary.csv")


=== TEST RESULTS (mean ± std) ===
Model  PSNR: 30.99 ± 2.05 dB
Model  SSIM: 0.8637 ± 0.0958
Model  LPİPS: 0.1070 ± 0.0823
Model  DISTS: 0.1175 ± 0.0230
Model  R_H: 0.6391 ± 0.2067
Test samples: 220


,psnr_pred_mean,psnr_pred_std,ssim_pred_mean,ssim_pred_std,lpips_pred_mean,lpips_pred_std,dists_pred_mean,dists_pred_std,r_h_pred_mean,r_h_pred_std,n_test,total_time
0,30.985754,2.048071,0.863679,0.095819,0.107029,0.082343,0.117507,0.022955,0.639131,0.206728,220,390.966085


### Evaluating Baseline Metrics (Noisy vs Ground Truth)

In [16]:
import pandas as pd
from tqdm import tqdm
from metrics import slice_metrics

baseline_metrics = []

# Iterate through the test loader to calculate metrics on raw inputs
for noisy_batch, gt_batch, _ in tqdm(test_loader, desc="Evaluating Baseline"):
    for i in range(noisy_batch.size(0)):
        noisy_2d = noisy_batch[i].squeeze().cpu().numpy()
        gt_2d = gt_batch[i].squeeze().cpu().numpy()

        # Compute metrics between noisy input and ground truth target
        metrics = slice_metrics(noisy_2d, gt_2d)
        baseline_metrics.append(metrics)

# Convert to DataFrame and summarize
df_baseline = pd.DataFrame(baseline_metrics)
save_per_slice_metrics(df_baseline, CONFIG.run_dir / "baseline_metrics.csv")
baseline_summary = {
    "psnr_raw_mean":  float(df_baseline["psnr"].mean()),
    "psnr_raw_std":   float(df_baseline["psnr"].std(ddof=1)),
    "ssim_raw_mean":  float(df_baseline["ssim"].mean()),
    "ssim_raw_std":   float(df_baseline["ssim"].std(ddof=1)),
    "lpips_raw_mean":  float(df_baseline["lpips"].mean()),
    "lpips_raw_std":   float(df_baseline["lpips"].std(ddof=1)),
    "dists_raw_mean":  float(df_baseline["dists"].mean()),
    "dists_raw_std":   float(df_baseline["dists"].std(ddof=1)),
    "r_h_raw_mean":  float(df_baseline["r_h"].mean()),
    "r_h_raw_std":   float(df_baseline["r_h"].std(ddof=1)),
    "n_test": len(df_baseline)
}

print("\n=== RAW NOISY INPUT TEST RESULTS (mean ± std) ===")
print(f"Raw PSNR: {baseline_summary['psnr_raw_mean']:.2f} ± {baseline_summary['psnr_raw_std']:.2f} dB")
print(f"Raw SSIM: {baseline_summary['ssim_raw_mean']:.4f} ± {baseline_summary['ssim_raw_std']:.4f}")
print(f"Raw LPIPS: {baseline_summary['lpips_raw_mean']:.4f} ± {baseline_summary['lpips_raw_std']:.4f}")
print(f"Raw DISTS: {baseline_summary['dists_raw_mean']:.4f} ± {baseline_summary['dists_raw_std']:.4f}")
print(f"Raw R_H: {baseline_summary['r_h_raw_mean']:.4f} ± {baseline_summary['r_h_raw_std']:.4f}")

save_per_slice_metrics([baseline_summary], CONFIG.run_dir / "baseline_summary.csv")



=== RAW NOISY INPUT TEST RESULTS (mean ± std) ===
Raw PSNR: 23.04 ± 2.13 dB
Raw SSIM: 0.7423 ± 0.0647
Raw LPIPS: 0.2843 ± 0.0959
Raw DISTS: 0.1641 ± 0.0462
Raw R_H: 6.1743 ± 5.8560


,psnr_raw_mean,psnr_raw_std,ssim_raw_mean,ssim_raw_std,lpips_raw_mean,lpips_raw_std,dists_raw_mean,dists_raw_std,r_h_raw_mean,r_h_raw_std,n_test
0,23.039831,2.134886,0.742302,0.064748,0.284252,0.095883,0.164106,0.046221,6.174314,5.856038,220


In [ ]:
from train import only_test
"""
df_test = only_test(CONFIG, model, CONFIG.run_dir/"scoped.pth", device)

summary = {
    "psnr_pred_mean":  float(df_test["psnr"].mean()),
    "psnr_pred_std":   float(df_test["psnr"].std(ddof=1)),
    "ssim_pred_mean":  float(df_test["ssim"].mean()),
    "ssim_pred_std":   float(df_test["ssim"].std(ddof=1)),
    "lpips_pred_mean":  float(df_test["lpips"].mean()),
    "lpips_pred_std":   float(df_test["lpips"].std(ddof=1)),
    "dists_pred_mean":  float(df_test["dists"].mean()),
    "dists_pred_std":   float(df_test["dists"].std(ddof=1)),
    "r_h_pred_mean":  float(df_test["r_h"].mean()),
    "r_h_pred_std":   float(df_test["r_h"].std(ddof=1)),
    "n_test": int(len(df_test)),
    "total_time": total_time,
}

print("\n=== TEST RESULTS (mean \u00b1 std) ===")
print(f"Model  PSNR: {summary['psnr_pred_mean']:.2f} \u00b1 {summary['psnr_pred_std']:.2f} dB")
print(f"Model  SSIM: {summary['ssim_pred_mean']:.4f} \u00b1 {summary['ssim_pred_std']:.4f}")
print(f"Model  LP\u0130PS: {summary['lpips_pred_mean']:.4f} \u00b1 {summary['lpips_pred_std']:.4f}")
print(f"Model  DISTS: {summary['dists_pred_mean']:.4f} \u00b1 {summary['dists_pred_std']:.4f}")
print(f"Model  R_H: {summary['r_h_pred_mean']:.4f} \u00b1 {summary['r_h_pred_std']:.4f}")

print(f"Test samples: {summary['n_test']}")

save_per_slice_metrics([summary], CONFIG.run_dir / "summary_scoped.csv")
"""

In [ ]:
!pip install -q huggingface_hub

In [32]:
from huggingface_hub import hf_hub_download
import huggingface_hub
import os

huggingface_hub.login()

radimagenet_weights_path = hf_hub_download(repo_id="Lab-Rasool/RadImageNet", filename="ResNet50.pt")
print(f"Successfully downloaded RadImageNet weights to: {radimagenet_weights_path}")

Successfully downloaded RadImageNet weights to: /root/.cache/huggingface/hub/models--Lab-Rasool--RadImageNet/snapshots/14460ee4c1276f6925611a63aa9a54a05d39eae0/ResNet50.pt


In [35]:
from feature_nets import build_feature_net

random_tensor = torch.rand(64, 64)
random_tensor = random_tensor.unsqueeze(0).unsqueeze(0)

vggnet = build_feature_net("vgg19")
print(len(vggnet.features(random_tensor)))

radimagenet_net = build_feature_net("radimagenet", radimagenet_weights_path)
print(len(radimagenet_net.features(random_tensor)))

dinov2_s_net = build_feature_net("dinov2_s")
print(len(dinov2_s_net.features(random_tensor)))

medsiglipnet = build_feature_net("medsiglip")
print(len(medsiglipnet.features(random_tensor)))

clip_b16net = build_feature_net("clip_b16")
print(len(clip_b16net.features(random_tensor)))

1
3


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


3


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

3


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

3
